***Importing modules***

In [1]:
!pip install optuna category_encoders catboost
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from category_encoders import TargetEncoder
import lightgbm as lgb
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 17.7 MB/s eta 0:00:00


***Loading datasets***

In [2]:
train = pd.read_csv('/content/drive/MyDrive/AL ML BootCamp Data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/AL ML BootCamp Data/test.csv')

target = train['total_sales']
train_features = train.drop(columns=['total_sales'])

***Combining both datasets***

In [3]:
combined = pd.concat([train_features, test], ignore_index=True)

***Cleaning categorical strings***

In [4]:
combined['fat_content'] = combined['fat_content'].str.lower().str.strip()
combined['product_category'] = combined['product_category'].str.lower().str.strip()

***Grouping transformtions to prevent global mean skewing***

In [5]:
combined['product_weight_kg'] = combined.groupby('product_category')['product_weight_kg'].transform(lambda x: x.fillna(x.mean()))
store_size_mode = combined.groupby('store_format', observed=False)['store_size'].transform(lambda x: x.mode()[0] if not x.mode().empty else "Medium")
combined['store_size'] = store_size_mode

***Fixing zero anomalies***

In [6]:
combined['shelf_visibility'] = combined['shelf_visibility'].replace(0.0, np.nan)
combined['shelf_visibility'] = combined.groupby('product_code')['shelf_visibility'].transform(lambda x: x.fillna(x.mean()))
combined['shelf_visibility'] = combined['shelf_visibility'].fillna(combined['shelf_visibility'].mean())


***Extracting patterns from structural keys***

In [7]:
combined['product_type_prefix'] = combined['product_code'].str.split('-').str[1]
combined['store_code_prefix'] = combined['store_code'].str.split('-').str[1]

***Feature engineering : pricing elasticity and interaction ratios***

In [8]:
combined['price_to_weight'] = combined['product_price'] / (combined['product_weight_kg'] + 0.1)
combined['store_mean_price'] = combined.groupby('store_code')['product_price'].transform('mean')
combined['price_relative_to_store'] = combined['product_price'] / combined['store_mean_price']

***Scaling visibility distribution***

In [9]:
mean_vis_store = combined.groupby('store_code')['shelf_visibility'].transform('mean')
combined['visibility_store_average'] = combined['shelf_visibility'] / mean_vis_store
mean_vis_cat = combined.groupby('product_category')['shelf_visibility'].transform('mean')
combined['visibility_to_cat_average'] = combined['shelf_visibility'] / mean_vis_cat

***Micro-market density interactions***

In [10]:
combined['store_product_count'] = combined.groupby('store_code')['product_code'].transform('count')

***Encoding ordinal tier mappings explicitly***

In [11]:
tier_mapping = {'Tier_1': 1, 'Tier_2': 2, 'Tier_3': 3}
combined['store_location_tier'] = combined['store_location_tier'].map(tier_mapping)

***Convert features to explicit structural category representations***

In [12]:
categorical_cols = ['product_type_prefix', 'store_code_prefix', 'product_category', 'fat_content', 'store_format', 'store_size']
for col in categorical_cols:
    combined[col] = combined[col].astype('category')

***Separate unified dataset back into analytical frames***

In [13]:
train_df = combined.iloc[:len(train)].copy()
test_df = combined.iloc[len(train):].copy()

***Add log transform target to training set to penalize proportional skew***

In [14]:
train_df['target_log'] = np.log1p(target)

***ROBUST TARGET ENCODING (LEAKAGE-FREE WITH 5-FOLD OUT-OF-FOLD)***

In [15]:
columns_to_encode = ['product_category', 'store_code']
kf = KFold(n_splits=5, shuffle=True, random_state=42)

 ***Create placeholder columns for out-of-fold representations***

In [16]:
for col in columns_to_encode:
    train_df[f'{col}_encoded'] = np.nan
    test_df[f'{col}_encoded'] = np.nan

***Compute targeted statistics strictly inside Out-Of-Fold constraints***

In [17]:
for train_idx, val_idx in kf.split(train_df):
    X_tr, X_va = train_df.iloc[train_idx], train_df.iloc[val_idx]

***Calculate conditional distribution stats only using the fold's training set***

In [18]:
 for col in columns_to_encode:
        means = X_tr.groupby(col, observed=False)['target_log'].mean()
        train_df.iloc[val_idx, train_df.columns.get_loc(f'{col}_encoded')] = X_va[col].map(means).astype(float)

***Map the global means to test data from untainted train distribution baseline***

In [19]:
for col in columns_to_encode:
    global_means = train_df.groupby(col, observed=False)['target_log'].mean()
    test_df[f'{col}_encoded'] = test_df[col].map(global_means).astype(float)
    train_df[f'{col}_encoded'] = train_df[f'{col}_encoded'].fillna(train_df['target_log'].mean())
    test_df[f'{col}_encoded'] = test_df[f'{col}_encoded'].fillna(train_df['target_log'].mean())

***Prepare final model feature matrices***

In [20]:
drop_cols = ['id', 'product_code', 'store_code']
X_train_full = train_df.drop(columns=drop_cols + ['target_log'])
X_test = test_df.drop(columns=drop_cols)
y_train_full_log = train_df['target_log']

 ***One-hot encoding preparation for models that don't take the pandas category dtype natively (XGBoost)***

In [21]:
X_train_full_encoded = pd.get_dummies(X_train_full)
X_test_encoded = pd.get_dummies(X_test)

***Ensure columns match between datasets post-dummy transformation***

In [22]:
X_train_full_encoded, X_test_encoded = X_train_full_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

 ***MULTI-MODEL 5-FOLD ENSEMBLE TRAINING & BLENDING LOOP***

In [23]:
lgb_test_preds = np.zeros(len(X_test))
xgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

In [24]:
print("Starting 5-Fold Cross-Validation Inference Loop across Ensemble Architectures...\n")

Starting 5-Fold Cross-Validation Inference Loop across Ensemble Architectures...



In [25]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
    print(f"--- Processing Fold {fold + 1} ---")

    y_tr, y_va = y_train_full_log.iloc[train_idx], y_train_full_log.iloc[val_idx]
    X_tr_lgb, X_va_lgb = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]

    lgb_params = {
        'n_estimators': 2000,
        'learning_rate': 0.01256,
        'num_leaves': 15,
        'min_child_samples': 47,
        'feature_fraction': 0.7947,
        'bagging_fraction': 0.6546,
        'bagging_freq': 2,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }

    model_lgb = LGBMRegressor(**lgb_params)
    model_lgb.fit(
        X_tr_lgb, y_tr,
        eval_set=[(X_va_lgb, y_va)],
        callbacks=[early_stopping(stopping_rounds=150, verbose=False)]
    )
    lgb_test_preds += model_lgb.predict(X_test) / kf.n_splits

    X_tr_xgb, X_va_xgb = X_train_full_encoded.iloc[train_idx], X_train_full_encoded.iloc[val_idx]

    model_xgb = XGBRegressor(
        n_estimators=2000,
        learning_rate=0.015,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=150
    )
    model_xgb.fit(
        X_tr_xgb, y_tr,
        eval_set=[(X_va_xgb, y_va)],
        verbose=False
    )
    xgb_test_preds += model_xgb.predict(X_test_encoded) / kf.n_splits

--- Processing Fold 1 ---
--- Processing Fold 2 ---
--- Processing Fold 3 ---
--- Processing Fold 4 ---
--- Processing Fold 5 ---


*** CatBoost works phenomenally well handling raw un-encoded object strings as category factors***

In [26]:
X_tr_cat, X_va_cat = X_train_full.iloc[train_idx].copy(), X_train_full.iloc[val_idx].copy()
X_test_cat = X_test.copy()

for c in categorical_cols:
    X_tr_cat[c] = X_tr_cat[c].astype(str)
    X_va_cat[c] = X_va_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)

In [27]:
cat_features_idx = [X_tr_cat.columns.get_loc(c) for c in categorical_cols]

model_cat = CatBoostRegressor(
    iterations=2500,
    learning_rate=0.02,
    depth=5,
    eval_metric='RMSE',
    random_state=42,
    verbose=False,
    early_stopping_rounds=150
)
model_cat.fit(
    X_tr_cat, y_tr,
    cat_features=cat_features_idx,
    eval_set=(X_va_cat, y_va),
    use_best_model=True
)
cat_test_preds += model_cat.predict(X_test_cat) / kf.n_splits

print("\nEnsemble training across all folds successfully completed.")


Ensemble training across all folds successfully completed.


In [28]:
final_test_preds_log = (0.45 * lgb_test_preds) + (0.30 * cat_test_preds) + (0.25 * xgb_test_preds)

In [29]:
final_test_preds_log

array([5.89879184, 6.25447559, 6.03851372, ..., 5.93870805, 6.30799647,
       5.56236271])

In [30]:
final_test_preds_real = np.expm1(final_test_preds_log)
final_test_preds_real = np.clip(final_test_preds_real, 0, None)  # Prevent negative prediction metrics

In [31]:
submission_dataframe = pd.DataFrame({'id': test['id'],'total_sales': final_test_preds_real})

In [32]:
submission_dataframe.to_csv('final_predictions_ensemble.csv', index=False)
print("Successfully generated and saved 'final_predictions_ensemble.csv'!")

Successfully generated and saved 'final_predictions_ensemble.csv'!


In [33]:
print(submission_dataframe.head())

          id  total_sales
0  row_00009   363.596711
1  row_00015   519.336435
2  row_00019   418.269418
3  row_00020   411.336323
4  row_00023   359.960552


In [34]:
submission_dataframe.to_csv('final_predictions_ensemble.csv', index=False)
from google.colab import files
files.download('final_predictions_ensemble.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>